# Polymer Property Prediction — GNN + CapsNet (single-model pipeline)

**This is a from-scratch rewrite, not an incremental edit.** Per request, every non-GNN+CapsNet
training path has been removed: the XGBoost/LightGBM/CatBoost ensembles, Ridge, multi-kernel GPR,
Nystroem+KernelRidge, ExtraTrees, the graph-transformer member, and the NNLS/ridge meta-stacking
layer that combined them. **GNN + CapsNet is now the only model family**, trained in three stages
(self-supervised pretraining → multi-task warm-start → per-target specialists) instead of being one
ingredient in an ensemble.

Two things changed to compensate for dropping the other models rather than just deleting code:

1. **Feature extraction is pared down to match what a GNN actually needs.** The ~5,000-column
   tree-oriented feature matrix (2D RDKit descriptors, Morgan/Avalon/MACCS fingerprints, fragment
   census, atom-pair/torsion/RDKit fingerprints, Kuenneth triplets) is gone — the GNN already sees
   the full atom/bond graph, so re-deriving the same structural information as a fixed-width vector
   for a tree model is redundant. What's kept is the **compact physics vector** (~22 dims: backbone
   conjugation length/fraction between the two `*` attachment points, aromaticity/ring/heteroatom
   density, TPSA, the Lorentz–Lorenz molar-volume proxy) plus the **cross-target sibling lookup**
   (other known properties of the same molecule) — both are cheap, molecule-level physics/chemistry
   signals that a 4-hop message-passing net cannot easily rediscover on its own, and both measurably
   helped in the earlier tree-ensemble version. These are now fed in as a small **global feature
   vector**, concatenated into the CapsNet head alongside the routed capsule output.
2. **The 1,000,000 unlabeled PI1M SMILES are now used for real, for every target.** The previous
   notebook only used PI1M for LightGBM-based pseudo-labeling of two targets (`ei`/`eea`). With no
   tree model left to generate pseudo-labels, PI1M is used instead for **self-supervised pretraining
   of the GNN+CapsNet encoder itself** — predicting a bank of cheap RDKit descriptors from molecular
   structure alone, at full 1M scale. That teaches the message-passing + capsule-routing trunk what
   "normal" polymer-repeat-unit chemistry looks like before it ever sees a labeled row, which helps
   **all seven** properties, not just two, and needs no auxiliary model.

**Pipeline, in order:** graph construction → compact physics/cross-target features → GNN+CapsNet
architecture → SSL pretraining on 1M PI1M molecules → multi-task supervised warm-start across all 7
properties → per-target specialist fine-tuning (the final model) → calibrated physics gate (kept
from before — it's a 1-parameter linear blend, not a competing model family) → submission.

**Runtime knobs** (top of the relevant cells): `PI1M_N` (SSL pretraining pool size, default the
full ~1,000,000), `SSL_EPOCHS`, `N_FOLDS`, and each stage's `epochs`/`patience`. The SSL stage over
the full PI1M pool is the single most expensive cell in the notebook — reduce `PI1M_N` for quick
iteration, and use the full value for a final run.


---

## v2 changes (in response to a real run: mean OOF R2 = 0.878, weakest on ei/eps/eea)

1. **Seed-ensembled specialists.** Each per-target specialist now trains with 2 seeds
   (different fold partition + different capsule-routing init each time) and averages OOF/test
   predictions — capsule routing has more init-to-init variance than plain pooling, so this is
   near-free variance reduction (`SPECIALIST_SEEDS`).
2. **Per-target epoch/patience budgets.** `eps` and `ei` (the two weakest targets) get a longer
   training budget (`PER_TARGET_EPOCHS`, `PER_TARGET_PATIENCE`) instead of one global default.
3. **Bigger capsule readout for the electronic-structure targets.** `ei`/`eea`/`egc` (all
   Koopmans-linked) get more capsules, larger pose vectors, and more routing iterations
   (`PER_TARGET_CAPS`) than the shared default.
4. **Two-term physics gate for eps/nc.** Previously eps~nc used only the Maxwell relation
   (`eps=nc²`) as the identity feature; it now also includes the Clausius-Mossotti/Lorentz-Lorenz
   form as a second, physically distinct nonlinear estimate, giving RidgeCV two real functional
   forms to blend instead of one. `PHYS_RECIPES` is now a list-of-functions format so any recipe
   can carry multiple derived terms.
5. **Gasteiger partial-charge statistics added to the SSL pretraining target bank.** None of the
   original ~28 SSL descriptors touch electronic structure, which plausibly under-serves
   ei/eea/egc relative to bulk/thermal properties like tg; charge distribution (mean/std/max/min
   over atoms) is a cheap structure-only proxy for electron-donating/withdrawing character.
6. **Longer SSL pretraining with a cosine LR schedule** (`SSL_EPOCHS` 3 → 6) instead of a fixed
   LR for a single light pass over the 1M-molecule pool.
7. **SMILES augmentation for SMALL_TARGETS.** A few random (non-canonical) atom orderings per
   training molecule are added to the training side of each fold (never validation/test, never
   crossing a fold boundary) via `AUG_PER_SMALL_TARGET` — standard low-cost augmentation for
   message-passing nets on scarce data.

None of this touches the model family (still GNN+CapsNet only) or the honest OOF evaluation
protocol — these are budget/architecture/feature knobs on the existing pipeline, not new models.


## 1. Setup

In [ ]:
import importlib, subprocess, sys

def ensure(import_name, pip_name=None):
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {pip_name or import_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name], check=True)

for imp, pip in [("rdkit", "rdkit"), ("torch", "torch")]:
    ensure(imp, pip)

import os, warnings, time, gc, math
import numpy as np, pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.linear_model import RidgeCV
import torch, torch.nn as nn, torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence

from rdkit import Chem
from rdkit.Chem import Crippen, Descriptors, rdMolDescriptors, AllChem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Imports OK. Torch device:", DEVICE)


## 2. Data Loading & Schema Normalization

In [ ]:
DATA_DIR = "/kaggle/input/competitions/ppp-round-2"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test.csv")
PI1M_PATH  = os.path.join(DATA_DIR, "PI1M.csv")

for _cand in ("train.csv", "/mnt/user-data/uploads/train.csv"):
    if not os.path.exists(TRAIN_PATH) and os.path.exists(_cand):
        TRAIN_PATH, TEST_PATH = _cand, _cand.replace("train.csv", "test.csv")
for _cand in ("PI1M.csv", "/mnt/user-data/uploads/PI1M.csv"):
    if not os.path.exists(PI1M_PATH) and os.path.exists(_cand): PI1M_PATH = _cand

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

def normalize_schema(df, is_train):
    df = df.copy(); low = {c.lower(): c for c in df.columns}
    idc = next((low[c] for c in ["id", "index"] if c in low), None)
    if idc is None: df.insert(0, "id", np.arange(len(df)))
    elif idc != "id": df = df.rename(columns={idc: "id"})
    sc = next((low[c] for c in ["smiles", "smile", "canonical_smiles"] if c in low), None)
    if sc is None: raise ValueError(f"No SMILES column in {list(df.columns)}")
    if sc != "smiles": df = df.rename(columns={sc: "smiles"})
    ttc = next((low[c] for c in ["target_type", "property", "property_type", "task"] if c in low), None)
    if ttc and ttc != "target_type": df = df.rename(columns={ttc: "target_type"})
    if "target_type" in df.columns:
        df["target_type"] = df["target_type"].astype(str).str.strip().str.lower()
    if is_train and "target" not in df.columns:
        low = {c.lower(): c for c in df.columns}
        tc = next((low[c] for c in ["target", "value", "y"] if c in low), None)
        if tc != "target": df = df.rename(columns={tc: "target"})
    return df

train = normalize_schema(train, True)
test  = normalize_schema(test, False)

TARGETS = sorted(train["target_type"].unique().tolist())
print("Discovered TARGETS:", TARGETS)

_counts = train["target_type"].value_counts().to_dict()
SMALL_THRESHOLD = 600
BIG_TARGETS   = [t for t in TARGETS if _counts.get(t, 0) >= SMALL_THRESHOLD]
SMALL_TARGETS = [t for t in TARGETS if _counts.get(t, 0) <  SMALL_THRESHOLD]
print(f"BIG targets: {BIG_TARGETS} | SMALL targets: {SMALL_TARGETS}")


## 2b. Canonicalization & Leak-Safe Group Folds

In [ ]:
def canonical(smi):
    m = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(m) if m is not None else None

for df in (train, test):
    df["smiles_canon"] = df["smiles"].apply(canonical)
    df["smiles_canon"] = df["smiles_canon"].fillna(df["smiles"])

bad = train["smiles"].apply(lambda s: Chem.MolFromSmiles(str(s)) is None)
if bad.any(): train = train[~bad].reset_index(drop=True)

key = ["smiles_canon", "target_type"]
grp = train.groupby(key)["target"]
spread = grp.transform(lambda s: s.max() - s.min())
scale = train.groupby("target_type")["target"].transform(lambda s: s.std())
bad_conflict = (spread > 0) & (spread > 0.5 * scale)
train = train[~bad_conflict].reset_index(drop=True)

train["target"] = train.groupby(key)["target"].transform("median")
train = train.drop_duplicates(subset=key, keep="first").reset_index(drop=True)
train["row_id"] = np.arange(len(train)); test["row_id"] = np.arange(len(test))

N_FOLDS = 8
def make_group_folds(groups, n_splits, seed=SEED):
    groups = np.asarray(groups)
    uniq, sizes = np.unique(groups, return_counts=True)
    rng = np.random.RandomState(seed); perm = rng.permutation(len(uniq))
    uniq, sizes = uniq[perm], sizes[perm]
    order = np.argsort(-sizes); load = np.zeros(n_splits, dtype=int); g2f = {}
    for gi in order:
        f = int(np.argmin(load)); g2f[uniq[gi]] = f; load[f] += sizes[gi]
    fid = np.array([g2f[g] for g in groups]); idx = np.arange(len(groups))
    return [(idx[fid != f], idx[fid == f]) for f in range(n_splits)]


## 3. Molecular Graph Construction

In [ ]:
ATOM_LIST = ["C", "N", "O", "S", "F", "Si", "P", "Cl", "Br", "I", "B", "H", "*"]
ATOM_MAP  = {s: i for i, s in enumerate(ATOM_LIST)}
HYB = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
       Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
       Chem.rdchem.HybridizationType.SP3D2]
HYB_MAP = {h: i for i, h in enumerate(HYB)}
BONDS = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
         Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
BT_MAP = {b: i for i, b in enumerate(BONDS)}

def _onehot(i, n):
    v = [0.0] * n
    if i is not None and 0 <= i < n: v[i] = 1.0
    return v

def atom_feat(a):
    return (_onehot(ATOM_MAP.get(a.GetSymbol()), len(ATOM_LIST)) + _onehot(min(a.GetDegree(), 5), 6)
            + _onehot(min(a.GetTotalNumHs(), 4), 5) + _onehot(HYB_MAP.get(a.GetHybridization()), len(HYB))
            + [float(a.GetIsAromatic()), float(a.IsInRing()),
               float(a.GetFormalCharge()), float(a.GetSymbol() == "*")])

def bond_feat(b):
    return _onehot(BT_MAP.get(b.GetBondType()), len(BONDS)) + [float(b.GetIsConjugated()), float(b.IsInRing())]

_m = Chem.MolFromSmiles("CC"); ADIM = len(atom_feat(_m.GetAtomWithIdx(0))); BDIM = len(bond_feat(_m.GetBondWithIdx(0)))

def to_graph(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None or m.GetNumAtoms() == 0: return None
    x = np.array([atom_feat(a) for a in m.GetAtoms()], dtype=np.float32)
    s, d, e = [], [], []
    for b in m.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); bf = bond_feat(b)
        s += [i, j]; d += [j, i]; e += [bf, bf]
    if not s: s, d, e = [0], [0], [[0.0] * BDIM]
    return x, np.array([s, d], dtype=np.int64), np.array(e, dtype=np.float32)

G_tr = [to_graph(s) for s in train["smiles"]]
G_te = [to_graph(s) for s in test["smiles"]]

def collate(graphs):
    xs, eis, eas, batch = [], [], [], []; off = 0
    for gi, g in enumerate(graphs):
        x, ei, ea = g; xs.append(x); eis.append(ei + off); eas.append(ea)
        batch += [gi] * x.shape[0]; off += x.shape[0]
    return (torch.tensor(np.concatenate(xs)),
            torch.tensor(np.concatenate(eis, axis=1)),
            torch.tensor(np.concatenate(eas)),
            torch.tensor(batch, dtype=torch.int64))

print(f"Graphs built: {len(G_tr)} train / {len(G_te)} test | ADIM={ADIM} BDIM={BDIM}")


## 3b. Compact Physics Feature Vector + Cross-Target Lookups

In [ ]:
def backbone_feats(m, smi):
    """~22-dim molecule-level physics/chemistry vector. Kept from the earlier tree-ensemble
    version because it carries information a 4-hop message-passing net does not easily derive
    on its own: backbone conjugation length between the two `*` attachment points spans the
    whole repeat unit (median backbone length in this dataset is well beyond a 4-hop receptive
    field), and the Lorentz-Lorenz molar-volume proxy is a closed-form physical relation, not a
    structural pattern to be learned from examples."""
    stars = [a.GetIdx() for a in m.GetAtoms() if a.GetSymbol() == "*"]
    path_len, conj_frac = np.nan, np.nan
    if len(stars) >= 2:
        try:
            p = Chem.GetShortestPath(m, stars[0], stars[1])
            if p and len(p) >= 2:
                path_len = len(p) - 1
                nc = sum(1 for i in range(len(p) - 1)
                         if (b := m.GetBondBetweenAtoms(p[i], p[i + 1])) is not None and b.GetIsConjugated())
                conj_frac = nc / path_len
        except Exception:
            pass
    na = m.GetNumAtoms()
    n_ar = sum(a.GetIsAromatic() for a in m.GetAtoms())
    n_hbd = rdMolDescriptors.CalcNumHBD(m); n_hba = rdMolDescriptors.CalcNumHBA(m)
    mw = Descriptors.MolWt(m); n_ring = rdMolDescriptors.CalcNumRings(m)
    n_rot = rdMolDescriptors.CalcNumRotatableBonds(m); fsp3 = rdMolDescriptors.CalcFractionCSP3(m)
    n_heavy = m.GetNumHeavyAtoms() or 1; tpsa = Descriptors.TPSA(m)
    try:
        mr = Crippen.MolMR(m)
        ll_proxy = mr / (mw / n_heavy)
    except Exception:
        ll_proxy = 0.0
    return [n_ar, sum(a.GetSymbol() == "O" for a in m.GetAtoms()),
            sum(a.GetSymbol() == "N" for a in m.GetAtoms()),
            sum(a.GetSymbol() == "S" for a in m.GetAtoms()),
            str(smi).count("*"), n_ar / na if na else 0.0,
            n_rot, n_ring, fsp3, path_len, conj_frac,
            n_hbd, n_hba, mw, rdMolDescriptors.CalcNumAromaticRings(m),
            (n_hbd + n_hba) / n_heavy, n_rot / n_heavy,
            n_ring / n_heavy, tpsa, tpsa / n_heavy,
            mw / (path_len + 1) if not np.isnan(path_len) else np.nan,
            ll_proxy]

EXTRA = ["n_arom", "n_O", "n_N", "n_S", "n_star", "arom_frac", "n_rotbond",
         "n_rings", "fsp3", "bb_path_len", "bb_conj_frac", "n_hbd", "n_hba", "mw",
         "n_arom_ring", "hbond_density", "rotbond_density", "ring_density",
         "tpsa", "tpsa_density", "mw_per_backbone", "lorentz_lorenz_proxy"]

def physics_vec(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None: return [0.0] * len(EXTRA)
    v = backbone_feats(m, smi)
    return [0.0 if (x is None or (isinstance(x, float) and np.isnan(x))) else float(x) for x in v]

print("Computing physics feature vectors ...")
PHYS_tr_raw = np.array([physics_vec(s) for s in train["smiles"]], dtype=np.float32)
PHYS_te_raw = np.array([physics_vec(s) for s in test["smiles"]], dtype=np.float32)
_pm, _ps = PHYS_tr_raw.mean(0), PHYS_tr_raw.std(0) + 1e-6
PHYS_tr = np.nan_to_num((PHYS_tr_raw - _pm) / _ps).astype(np.float32)
PHYS_te = np.nan_to_num((PHYS_te_raw - _pm) / _ps).astype(np.float32)

# --- cross-target sibling lookup: other known properties of the same molecule ---
# Simplified vs. the earlier LightGBM-imputed version: unknown entries are filled with the
# training median and flagged via a "known" bit, and the GNN+CapsNet head learns directly
# whether/how to use each column -- no auxiliary imputation model needed.
T = len(TARGETS)
_tidx = {t: i for i, t in enumerate(TARGETS)}
_canon_tr = train["smiles_canon"].values; _tt_tr = train["target_type"].values
_y_tr = train["target"].values.astype(np.float32)
_canon_te = test["smiles_canon"].values;  _tt_te = test["target_type"].values

_LUT = {t: {} for t in TARGETS}
for c, t, v in zip(_canon_tr, _tt_tr, _y_tr): _LUT[t][c] = v
_MED = {t: (float(np.median(list(_LUT[t].values()))) if _LUT[t] else 0.0) for t in TARGETS}

def _cross_block(canon):
    n = len(canon)
    vals = np.zeros((n, T), np.float32); known = np.zeros((n, T), np.float32)
    for j, t in enumerate(TARGETS):
        lut = _LUT[t]
        for i, c in enumerate(canon):
            if c in lut: vals[i, j] = lut[c]; known[i, j] = 1.0
            else: vals[i, j] = _MED[t]
    return vals, known

CROSS_tr_val, CROSS_tr_known = _cross_block(_canon_tr)
CROSS_te_val, CROSS_te_known = _cross_block(_canon_te)

# a row's own target must never appear as a "sibling" feature -- zero its own column out
for i in range(len(_tt_tr)):
    j = _tidx[_tt_tr[i]]; CROSS_tr_val[i, j] = _MED[TARGETS[j]]; CROSS_tr_known[i, j] = 0.0
for i in range(len(_tt_te)):
    j = _tidx[_tt_te[i]]; CROSS_te_val[i, j] = _MED[TARGETS[j]]; CROSS_te_known[i, j] = 0.0

GLOBAL_tr = np.hstack([PHYS_tr, CROSS_tr_val, CROSS_tr_known]).astype(np.float32)
GLOBAL_te = np.hstack([PHYS_te, CROSS_te_val, CROSS_te_known]).astype(np.float32)
GDIM = GLOBAL_tr.shape[1]
print(f"Global feature vector dim: {GDIM}  (physics {PHYS_tr.shape[1]} + cross-value {T} + cross-known {T})")


## 4. GNN + CapsNet Architecture (single model family)

In [ ]:
CAPS_DIM      = 16     # dim of each capsule's pose vector
NUM_CAPS      = 10     # number of molecule-level output capsules
ROUTING_ITERS = 3
ENCODER_PREFIXES = ("lin0.", "edge.", "msg.", "upd.", "bn.", "caps.")  # shared trunk, warm-startable


def squash(s, dim=-1, eps=1e-8):
    sq = (s * s).sum(dim=dim, keepdim=True)
    return (sq / (1.0 + sq)) * s / torch.sqrt(sq + eps)


def to_dense(h, B, ng):
    """Un-flatten the (total_atoms, hid) tensor from `collate` into a padded (ng, maxN, hid)
    tensor + validity mask. Relies on `collate` laying atoms out in contiguous per-graph blocks
    in graph order, so a plain split + pad reconstructs per-graph atom sets with no gather/scatter."""
    counts = torch.bincount(B, minlength=ng).tolist()
    parts = torch.split(h, counts, dim=0)
    dense = pad_sequence(parts, batch_first=True)
    mask = pad_sequence([torch.ones(c, device=h.device) for c in counts], batch_first=True)
    return dense, mask


class CapsuleReadout(nn.Module):
    """Atoms -> primary capsules -> routed-by-agreement molecule-level capsules. Each atom votes,
    for every output capsule, what that capsule's pose should be; routing-by-agreement (softmax +
    squash, iterated) keeps only the votes that agree, instead of a single learned attention score."""
    def __init__(self, hid, caps_dim=CAPS_DIM, num_out=NUM_CAPS, iters=ROUTING_ITERS):
        super().__init__()
        self.in_proj = nn.Linear(hid, caps_dim)
        self.num_out, self.caps_dim, self.iters = num_out, caps_dim, iters
        self.W = nn.Parameter(0.01 * torch.randn(num_out, caps_dim, caps_dim))

    def forward(self, dense_h, mask):
        u = squash(self.in_proj(dense_h))                     # (B,N,Din) primary capsules
        u_hat = torch.einsum('bnd,odf->bnof', u, self.W)       # (B,N,O,Dout) votes
        Bsz, N, O, Dout = u_hat.shape
        b = torch.zeros(Bsz, N, O, device=dense_h.device)
        maskf = mask.unsqueeze(-1)
        v = None
        for it in range(self.iters):
            c = torch.softmax(b.masked_fill(maskf == 0, -1e9), dim=2) * maskf
            s = (c.unsqueeze(-1) * u_hat).sum(dim=1)           # (B,O,Dout)
            v = squash(s, dim=-1)
            if it < self.iters - 1:
                b = b + torch.einsum('bnof,bof->bno', u_hat, v)
        return v                                               # (B,O,Dout)


class GNNCapsNet(nn.Module):
    """GNN message-passing encoder + capsule-routing readout + global physics/cross-target
    feature fusion. `n_tasks` controls how many independent regression heads sit on top of the
    shared trunk: n_tasks=len(_SSL_COLS) for SSL pretraining, n_tasks=len(TARGETS) for the
    multi-task supervised warm-start, n_tasks=1 for a per-target specialist. Every stage shares
    the same `lin0/edge/msg/upd/bn/caps` trunk, so later stages can warm-start from earlier ones
    by copying only the ENCODER_PREFIXES tensors. caps_dim/num_out_caps/routing_iters let a
    specific target use a bigger or more-iterated capsule readout than the shared default; if
    they differ from the warm-start source's config, only the caps.* tensors are retrained from
    scratch on warm-start (the message-passing trunk is copied regardless, since its shape does
    not depend on the capsule config)."""
    def __init__(self, adim, bdim, gdim, hid=192, layers=4, drop=0.1, n_tasks=1,
                 caps_dim=CAPS_DIM, num_out_caps=NUM_CAPS, routing_iters=ROUTING_ITERS):
        super().__init__(); self.L = layers
        self.lin0 = nn.Linear(adim, hid); self.edge = nn.Linear(bdim, hid)
        self.msg = nn.ModuleList([nn.Linear(2 * hid, hid) for _ in range(layers)])
        self.upd = nn.ModuleList([nn.GRUCell(hid, hid) for _ in range(layers)])
        self.bn  = nn.ModuleList([nn.BatchNorm1d(hid) for _ in range(layers)])
        self.caps = CapsuleReadout(hid, caps_dim=caps_dim, num_out=num_out_caps, iters=routing_iters)
        in_dim = num_out_caps * caps_dim + gdim
        self.heads = nn.ModuleList([nn.Sequential(nn.Linear(in_dim, hid // 2), nn.ReLU(),
                                                   nn.Dropout(drop), nn.Linear(hid // 2, 1))
                                     for _ in range(n_tasks)])

    def encode(self, X, EI, EA, B):
        h = F.relu(self.lin0(X)); e = self.edge(EA); s, d = EI[0], EI[1]
        for l in range(self.L):
            msg = torch.relu(self.msg[l](torch.cat([h[s], e], 1)))
            agg = torch.zeros_like(h).index_add_(0, d, msg)
            h = self.bn[l](self.upd[l](agg, h))
        ng = int(B.max().item()) + 1
        dense_h, mask = to_dense(h, B, ng)
        return self.caps(dense_h, mask).flatten(1)      # (ng, NUM_CAPS*CAPS_DIM)

    def forward(self, X, EI, EA, B, G):
        z = torch.cat([self.encode(X, EI, EA, B), G], dim=1)
        return torch.cat([hd(z) for hd in self.heads], dim=1)


def copy_encoder(dst_model, state):
    """Warm-start only the shared trunk (lin0/edge/msg/upd/bn/caps) from a state dict produced
    by an earlier stage. Head weights are always trained fresh -- they're stage-specific."""
    if not state:
        return
    sd = dst_model.state_dict()
    matched = {k: v for k, v in state.items()
               if k.startswith(ENCODER_PREFIXES) and k in sd and sd[k].shape == v.shape}
    sd.update(matched); dst_model.load_state_dict(sd)


def copy_encoder_and_head(dst_model, state, src_head_idx=0, dst_head_idx=0):
    """Warm-start the shared trunk, plus one specific task's head into the destination model's
    head (used to seed a per-target specialist from the multi-task model's matching head)."""
    if not state:
        return
    sd = dst_model.state_dict()
    matched = {k: v for k, v in state.items()
               if k.startswith(ENCODER_PREFIXES) and k in sd and sd[k].shape == v.shape}
    src_prefix, dst_prefix = f"heads.{src_head_idx}.", f"heads.{dst_head_idx}."
    for k, v in state.items():
        if k.startswith(src_prefix):
            dk = dst_prefix + k[len(src_prefix):]
            if dk in sd and sd[dk].shape == v.shape:
                matched[dk] = v
    sd.update(matched); dst_model.load_state_dict(sd)


def _predict(model, graphs, globals_, bs=256, n_out=1):
    model.eval(); out = np.zeros((len(graphs), n_out), np.float32)
    with torch.no_grad():
        for i in range(0, len(graphs), bs):
            X, EI, EA, B = collate(graphs[i:i + bs])
            Gv = torch.tensor(globals_[i:i + bs])
            X, EI, EA, B, Gv = X.to(DEVICE), EI.to(DEVICE), EA.to(DEVICE), B.to(DEVICE), Gv.to(DEVICE)
            out[i:i + bs] = model(X, EI, EA, B, Gv).cpu().numpy()
    return out


## 5. Self-Supervised Pretraining on 1M Unlabeled PI1M SMILES

In [ ]:
PI1M_N     = 1_000_000   # use the full unlabeled PI1M pool, as requested; lower this for quick iteration
SSL_EPOCHS = 6           # bumped from 3 -- one pass over 1M molecules is a light touch; cosine LR below
SSL_BS     = 512
SSL_LR     = 1e-3

# cheap RDKit descriptors as the SSL regression target -- structure alone must predict these,
# which forces the encoder to learn general-purpose polymer chemistry before it ever sees a label
_SSL_COLS = ["MolWt", "MolLogP", "MolMR", "TPSA", "NumRotatableBonds", "RingCount", "NumAromaticRings",
             "FractionCSP3", "NumHAcceptors", "NumHDonors", "HeavyAtomCount", "NHOHCount", "NOCount",
             "NumAliphaticRings", "NumSaturatedRings", "BalabanJ", "BertzCT", "Chi0v", "Chi1v", "Chi2v",
             "Kappa1", "Kappa2", "Kappa3", "HallKierAlpha", "LabuteASA", "qed", "NumHeteroatoms",
             "NumValenceElectrons"]
_SFN = {n: f for n, f in Descriptors.descList if n in set(_SSL_COLS)}
_SORD = [n for n in _SSL_COLS if n in _SFN]

# Gasteiger partial-charge statistics -- added because none of the descriptors above touch
# electronic structure, which is plausibly why the Koopmans-linked targets (ei/eea/egc) benefit
# less from SSL pretraining than a bulk/thermal property like tg. Charge distribution over the
# backbone is a cheap structure-only proxy for electron-donating/withdrawing character.
_GAST_COLS = ["gast_mean", "gast_std", "gast_max", "gast_min"]

def _gasteiger_stats(m):
    try:
        mc = Chem.Mol(m)
        AllChem.ComputeGasteigerCharges(mc)
        ch = np.array([float(a.GetProp("_GasteigerCharge")) for a in mc.GetAtoms()], dtype=np.float64)
        ch = ch[np.isfinite(ch)]
        if ch.size == 0: return [0.0, 0.0, 0.0, 0.0]
        return [float(ch.mean()), float(ch.std()), float(ch.max()), float(ch.min())]
    except Exception:
        return [0.0, 0.0, 0.0, 0.0]

_SSL_ALL_COLS = _SORD + _GAST_COLS

def _ssl_row(smi):
    g = to_graph(smi)
    if g is None: return None
    m = Chem.MolFromSmiles(str(smi))
    if m is None: return None
    r = []
    for n in _SORD:
        try:
            v = float(_SFN[n](m)); r.append(v if np.isfinite(v) else 0.0)
        except Exception:
            r.append(0.0)
    r += _gasteiger_stats(m)
    return g, np.asarray(r, np.float32), np.asarray(physics_vec(smi), np.float32)

SSL_ENCODER_STATE = None
if os.path.exists(PI1M_PATH):
    _t0 = time.time()
    _pdf = pd.read_csv(PI1M_PATH)
    _pc = "SMILES" if "SMILES" in _pdf.columns else _pdf.columns[0]
    _sm = _pdf[_pc].astype(str).drop_duplicates().values; del _pdf; gc.collect()
    n_avail = len(_sm); n_use = min(PI1M_N, n_avail)
    print(f"PI1M pool: {n_avail:,} unique SMILES available, using {n_use:,} for SSL pretraining")
    _sel = _sm[np.random.RandomState(SEED + 3).permutation(n_avail)[:n_use]]

    _rows = []
    for i, s in enumerate(_sel):
        r = _ssl_row(s)
        if r is not None: _rows.append(r)
        if (i + 1) % 100_000 == 0:
            print(f"  encoded {i + 1:,}/{n_use:,}  ({(time.time() - _t0) / 60:.1f} min)")
    print(f"  usable graphs: {len(_rows):,}/{n_use:,}  (prep {(time.time() - _t0) / 60:.1f} min)")

    if len(_rows) < 5000:
        print("  too few usable PI1M molecules -> skipping SSL pretraining, models train from scratch")
    else:
        G_ssl = [r[0] for r in _rows]
        Y_ssl = np.vstack([r[1] for r in _rows])
        PHYS_ssl_raw = np.vstack([r[2] for r in _rows])
        _med, (_q1, _q3) = np.median(Y_ssl, 0), np.percentile(Y_ssl, [25, 75], axis=0)
        Y_ssl = np.clip((Y_ssl - _med) / np.clip(_q3 - _q1, 1e-6, None), -8, 8).astype(np.float32)
        PHYS_ssl = np.nan_to_num((PHYS_ssl_raw - _pm) / _ps).astype(np.float32)
        # unlabeled molecules carry no sibling info: cross-value = training medians, cross-known = 0
        G_ssl_global = np.hstack([
            PHYS_ssl,
            np.tile(np.array([_MED[t] for t in TARGETS], np.float32), (len(_rows), 1)),
            np.zeros((len(_rows), T), np.float32),
        ]).astype(np.float32)

        ssl_model = GNNCapsNet(ADIM, BDIM, GDIM, n_tasks=len(_SSL_ALL_COLS)).to(DEVICE)
        opt = torch.optim.AdamW(ssl_model.parameters(), lr=SSL_LR, weight_decay=1e-6)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=SSL_EPOCHS)
        n = len(G_ssl); idx = np.arange(n)
        for ep in range(SSL_EPOCHS):
            ssl_model.train(); np.random.shuffle(idx); tot = 0.0; nbb = 0
            for k in range(0, n, SSL_BS):
                bi = idx[k:k + SSL_BS]
                if len(bi) < 2: continue
                X, EI, EA, B = collate([G_ssl[j] for j in bi])
                Gv = torch.tensor(G_ssl_global[bi])
                X, EI, EA, B, Gv = X.to(DEVICE), EI.to(DEVICE), EA.to(DEVICE), B.to(DEVICE), Gv.to(DEVICE)
                tgt = torch.tensor(Y_ssl[bi], device=DEVICE)
                pred = ssl_model(X, EI, EA, B, Gv)
                loss = F.smooth_l1_loss(pred, tgt)
                opt.zero_grad(); loss.backward(); opt.step()
                tot += loss.item(); nbb += 1
            sched.step()
            print(f"  SSL epoch {ep + 1}/{SSL_EPOCHS}  loss={tot / max(nbb, 1):.4f}  lr={sched.get_last_lr()[0]:.2e}")

        SSL_ENCODER_STATE = {k: v.detach().cpu().clone() for k, v in ssl_model.state_dict().items()
                              if k.startswith(ENCODER_PREFIXES)}
        print(f"Captured SSL-pretrained encoder ({len(SSL_ENCODER_STATE)} tensors) from "
              f"{len(G_ssl):,} unlabeled PI1M molecules")
        del G_ssl, Y_ssl, PHYS_ssl, G_ssl_global, ssl_model; gc.collect()
else:
    print("PI1M.csv not found -> skipping SSL pretraining, models will train from scratch")


## 6. Multi-Task Supervised Warm-Start (all 7 properties, shared trunk)

In [ ]:
_MT_MU = np.array([train.loc[train.target_type == t, "target"].mean() for t in TARGETS], np.float32)
_MT_SD = np.array([train.loc[train.target_type == t, "target"].std() + 1e-6 for t in TARGETS], np.float32)

def train_multitask(epochs=100, bs=64, lr=5e-4, patience=25):
    groups_all = train["smiles_canon"].values; tt = train["target_type"].values
    y = train["target"].values.astype(np.float32)
    ymat = np.full((len(train), T), np.nan, np.float32)
    for i in range(len(train)): ymat[i, _tidx[tt[i]]] = y[i]
    ynorm = (ymat - _MT_MU) / _MT_SD
    row_of = {t: np.where(tt == t)[0] for t in TARGETS}
    pos_in = {t: {r: i for i, r in enumerate(row_of[t])} for t in TARGETS}
    oof = {t: np.full(len(row_of[t]), np.nan, np.float32) for t in TARGETS}
    tst = {t: np.zeros(len(test), np.float32) for t in TARGETS}

    encoder_state = None
    for tr, va in make_group_folds(groups_all, N_FOLDS, seed=SEED):
        model = GNNCapsNet(ADIM, BDIM, GDIM, n_tasks=T).to(DEVICE)
        copy_encoder(model, SSL_ENCODER_STATE)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

        gtr = [G_tr[i] for i in tr]; Gtr_g = GLOBAL_tr[tr]; Ytr = ynorm[tr]
        Mtr = torch.tensor(~np.isnan(Ytr)); Ytr0 = torch.tensor(np.nan_to_num(Ytr))
        gva = [G_tr[i] for i in va]; Gva_g = GLOBAL_tr[va]
        idx = np.arange(len(tr)); best = -1e9; best_state = None; wait = 0

        for ep in range(epochs):
            model.train(); np.random.shuffle(idx)
            for k in range(0, len(idx), bs):
                bi = idx[k:k + bs]
                if len(bi) < 2: continue
                X, EI, EA, B = collate([gtr[j] for j in bi])
                Gv = torch.tensor(Gtr_g[bi])
                X, EI, EA, B, Gv = X.to(DEVICE), EI.to(DEVICE), EA.to(DEVICE), B.to(DEVICE), Gv.to(DEVICE)
                pred = model(X, EI, EA, B, Gv); mmask = Mtr[bi].to(DEVICE); tgt = Ytr0[bi].to(DEVICE)
                diff = (pred - tgt)[mmask]
                loss = F.smooth_l1_loss(diff, torch.zeros_like(diff))
                opt.zero_grad(); loss.backward(); opt.step()
            sched.step()
            vp = _predict(model, gva, Gva_g, n_out=T) * _MT_SD + _MT_MU
            va_list = list(va); r2s = []
            for j, t in enumerate(TARGETS):
                rows = [r for r in va if tt[r] == t]
                if len(rows) >= 5:
                    yr = y[rows]; pr = vp[[va_list.index(r) for r in rows], j]; r2s.append(r2_score(yr, pr))
            r2 = float(np.mean(r2s)) if r2s else -1e9
            if r2 > best:
                best = r2; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}; wait = 0
            else:
                wait += 1
                if wait >= patience: break

        if best_state is None:
            # validation R2 never improved on -1e9 (e.g. too few validation rows to score any
            # target this fold) -- fall back to the current weights instead of crashing.
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(best_state)
        vp = _predict(model, gva, Gva_g, n_out=T) * _MT_SD + _MT_MU
        for li, r in enumerate(va):
            t = tt[r]; j = _tidx[t]; p = pos_in[t][r]
            oof[t][p] = vp[li, j]
        encoder_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()
                          if k.startswith(ENCODER_PREFIXES)}
        tp = _predict(model, G_te, GLOBAL_te, n_out=T) * _MT_SD + _MT_MU
        for j, t in enumerate(TARGETS): tst[t] += tp[:, j] / N_FOLDS

    res = {}
    for t in TARGETS:
        yt = y[row_of[t]]; v = ~np.isnan(oof[t])
        print(f"[{t}] multitask OOF R2 = {r2_score(yt[v], oof[t][v]):.4f}")
        res[t] = {"oof": oof[t], "test": tst[t], "y": yt}
    return res, encoder_state

print("Training multi-task GNN+CapsNet warm-start ...")
mtl_res, MTL_ENCODER_STATE = train_multitask()


## 7. Per-Target Specialist Fine-Tuning (final model)

In [ ]:
# ── Per-target overrides ──────────────────────────────────────────────────────────────────────
# eps/ei were the weakest specialists in the last run (0.84 / 0.81 OOF R2 vs 0.86-0.93 elsewhere).
# Give them a longer training budget before deciding they've converged.
PER_TARGET_EPOCHS   = {"eps": 130, "ei": 130}
PER_TARGET_PATIENCE = {"eps": 28,  "ei": 28}

# ei/eea/egc are the Koopmans-linked electronic-structure targets -- give them a bigger, more-
# iterated capsule readout than the shared default. Since caps.* shapes then differ from the
# multi-task warm-start source, only those tensors retrain from scratch (see GNNCapsNet docstring
# in §4); the message-passing trunk warm-start is unaffected.
PER_TARGET_CAPS = {
    "ei":  dict(caps_dim=24, num_out_caps=14, routing_iters=5),
    "eea": dict(caps_dim=24, num_out_caps=14, routing_iters=5),
    "egc": dict(caps_dim=24, num_out_caps=14, routing_iters=5),
}

# SMILES augmentation for SMALL_TARGETS only: a few random (non-canonical) atom orderings of the
# same scarce training molecules, added to the training side of each fold only (never to
# validation/test, and never crossing a fold boundary -- see the group-matching logic below).
# This is standard low-cost augmentation for message-passing nets: it doesn't change the label,
# just gives the encoder more traversal orders of the same molecule to learn invariance from.
AUG_PER_SMALL_TARGET = 2

# Specialists are trained with multiple seeds (different group-fold partitions *and* different
# capsule-routing initialization) and averaged, since routing init adds more run-to-run variance
# than plain mean/attention pooling. Two seeds roughly doubles specialist training time; add a
# third to SPECIALIST_SEEDS for a bit more stability if the runtime budget allows.
SPECIALIST_SEEDS = [SEED, SEED + 101]


def random_smiles_variants(smi, n, max_tries=None):
    """Up to `n` distinct non-canonical SMILES strings for the same molecule (random atom order/
    root each call). Best-effort: returns fewer than n if the molecule doesn't have that many
    distinct textual forms, or [] if it fails to parse."""
    m = Chem.MolFromSmiles(str(smi))
    if m is None:
        return []
    max_tries = max_tries or (n * 10)
    out, tries = set(), 0
    while len(out) < n and tries < max_tries:
        tries += 1
        try:
            s = Chem.MolToSmiles(m, doRandom=True, canonical=False)
        except Exception:
            break
        out.add(s)
    return list(out)


def train_specialist(target, seed=SEED):
    is_small = target in SMALL_TARGETS
    epochs = PER_TARGET_EPOCHS.get(target, 90)
    patience = PER_TARGET_PATIENCE.get(target, 18)
    caps_over = PER_TARGET_CAPS.get(target, {})
    lr_e = 1.5e-4 if is_small else 3e-4
    bs = 32 if is_small else 96
    mask = (train["target_type"].values == target)
    rows = np.where(mask)[0]
    y = train.loc[mask, "target"].values.astype(np.float32)
    groups = train.loc[mask, "smiles_canon"].values
    gsub = [G_tr[r] for r in rows]; Gsub_g = GLOBAL_tr[rows]
    ok = [i for i, g in enumerate(gsub) if g is not None]
    if len(ok) < 0.9 * len(gsub):
        print(f"[{target}] only {len(ok)}/{len(gsub)} encodable -> skipping")
        return None

    # Build the augmentation pool once (outside the fold loop), keyed by each row's original
    # canonical group -- a fold loop below only pulls in augmented copies whose group is in that
    # fold's *training* set, so no augmented copy of a validation molecule can leak in.
    extra_g, extra_y, extra_groups, extra_Gg = [], [], [], []
    if is_small and AUG_PER_SMALL_TARGET > 0:
        smi_list = train.loc[mask, "smiles"].values
        for i in range(len(gsub)):
            if gsub[i] is None:
                continue
            for v in random_smiles_variants(smi_list[i], AUG_PER_SMALL_TARGET):
                gg = to_graph(v)
                if gg is None:
                    continue
                extra_g.append(gg); extra_y.append(y[i]); extra_groups.append(groups[i]); extra_Gg.append(Gsub_g[i])
        if extra_g:
            extra_y = np.array(extra_y, np.float32)
            extra_groups = np.array(extra_groups)
            extra_Gg = np.vstack(extra_Gg).astype(np.float32)
            print(f"[{target}] SMILES augmentation: +{len(extra_g)} atom-order variants "
                  f"from {len(gsub)} molecules")

    oof = np.full(len(y), np.nan, np.float32)
    tst = np.zeros(len(test), np.float32); nfold = 0
    torch.manual_seed(seed); np.random.seed(seed)
    j_src = _tidx[target]

    for tr_i, va_i in make_group_folds(groups, N_FOLDS, seed=seed):
        if len(tr_i) < 20 or len(va_i) < 2:
            continue
        mu, sd = y[tr_i].mean(), y[tr_i].std() + 1e-8
        gtr = [gsub[i] for i in tr_i]; Gtr_g = Gsub_g[tr_i]; ytr_raw = y[tr_i]

        if len(extra_g):
            tr_group_set = set(groups[tr_i])
            aug_idx = [k for k in range(len(extra_g)) if extra_groups[k] in tr_group_set]
            if aug_idx:
                gtr = gtr + [extra_g[k] for k in aug_idx]
                Gtr_g = np.vstack([Gtr_g, extra_Gg[aug_idx]])
                ytr_raw = np.concatenate([ytr_raw, extra_y[aug_idx]])

        ytr = (ytr_raw - mu) / sd
        gva = [gsub[i] for i in va_i]; Gva_g = Gsub_g[va_i]

        model = GNNCapsNet(ADIM, BDIM, GDIM, n_tasks=1, **caps_over).to(DEVICE)
        copy_encoder_and_head(model, MTL_ENCODER_STATE, src_head_idx=j_src, dst_head_idx=0)
        if is_small:
            # ~220 rows can overwrite an inherited trunk in a couple of epochs -- freeze the
            # first two message-passing layers, keeping the rest of the inherited encoder mobile.
            for p in model.lin0.parameters(): p.requires_grad_(False)
            for l in (0, 1):
                for p in model.msg[l].parameters(): p.requires_grad_(False)
                for p in model.upd[l].parameters(): p.requires_grad_(False)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                 lr=lr_e, weight_decay=1e-5)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

        idx = np.arange(len(gtr)); best = -1e9; best_state = None; wait = 0
        for ep in range(epochs):
            model.train(); np.random.shuffle(idx)
            for k in range(0, len(idx), bs):
                bi = idx[k:k + bs]
                if len(bi) < 2: continue
                X, EI, EA, B = collate([gtr[j] for j in bi])
                Gv = torch.tensor(Gtr_g[bi])
                X, EI, EA, B, Gv = X.to(DEVICE), EI.to(DEVICE), EA.to(DEVICE), B.to(DEVICE), Gv.to(DEVICE)
                pred = model(X, EI, EA, B, Gv).squeeze(-1)
                tgt = torch.tensor(ytr[bi], device=DEVICE, dtype=torch.float32)
                loss = F.smooth_l1_loss(pred, tgt)
                opt.zero_grad(); loss.backward(); opt.step()
            sched.step()
            vp = _predict(model, gva, Gva_g).squeeze(-1) * sd + mu
            r2 = r2_score(y[va_i], vp) if len(va_i) >= 5 else -1e9
            if r2 > best:
                best = r2; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}; wait = 0
            else:
                wait += 1
                if wait >= patience: break
        if best_state is None:
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(best_state)
        oof[va_i] = _predict(model, gva, Gva_g).squeeze(-1) * sd + mu
        tst += (_predict(model, G_te, GLOBAL_te).squeeze(-1) * sd + mu) / N_FOLDS
        nfold += 1

    v = np.isfinite(oof)
    if nfold == 0 or v.sum() < 5:
        print(f"[{target}] (seed={seed}) too few valid folds -> skipping")
        return None
    print(f"[{target}] (seed={seed}) GNN+CapsNet specialist OOF R2 = "
          f"{r2_score(y[v], oof[v]):.4f}  ({nfold}/{N_FOLDS} folds)")
    return {"oof": oof, "test": tst, "y": y, "mask": mask}


def train_specialist_multiseed(target, seeds=SPECIALIST_SEEDS):
    """Train `target` once per seed (different fold partition + different capsule init each
    time) and average. Uses nanmean on the OOF stack since a row may be validated in some seed
    runs and not others if a fold happened to fall below the minimum-row threshold that seed."""
    oof_stack, test_stack, y_ref, mask_ref = [], [], None, None
    for sd in seeds:
        r = train_specialist(target, seed=sd)
        if r is None:
            continue
        oof_stack.append(r["oof"]); test_stack.append(r["test"])
        y_ref, mask_ref = r["y"], r["mask"]
    if not oof_stack:
        print(f"[{target}] no seed produced a usable model -> skipping")
        return None
    oof_mean = np.nanmean(np.vstack(oof_stack), axis=0)
    test_mean = np.mean(np.vstack(test_stack), axis=0)
    v = np.isfinite(oof_mean)
    if v.sum() < 5:
        print(f"[{target}] too few valid rows after seed-averaging -> skipping")
        return None
    print(f"[{target}] seed-averaged ({len(oof_stack)}/{len(seeds)} seeds) "
          f"OOF R2 = {r2_score(y_ref[v], oof_mean[v]):.4f}")
    return {"oof": oof_mean, "test": test_mean, "y": y_ref, "mask": mask_ref}


print("Training per-target GNN+CapsNet specialists ...")
final_res = {}
for _t in TARGETS:
    _r = train_specialist_multiseed(_t)
    if _r is not None:
        final_res[_t] = _r


## 8. Calibrated Physics Head (sibling delta-learning, gated blend)

Kept unchanged in spirit from the earlier version: this is a 1-parameter linear calibration, not
a competing model, so it stays even though every other non-GNN+CapsNet component was removed.

On molecules where both sibling values are known, the raw physics identities are *qualitatively
right but quantitatively off* (e.g. `eps = nc^2` scores far higher after a 2-parameter linear
rescale than raw). Fitting that rescale explicitly with `RidgeCV` on ~dozens-to-hundreds of covered
rows is far more sample-efficient than asking the GNN to discover it implicitly from one noisy
input column. It is blended in per target with a CV-selected `alpha` that includes 0 in its grid,
so a target that gains nothing from it simply keeps its GNN+CapsNet prediction unchanged.


In [ ]:
def _maxwell_eps(nc):
    return nc ** 2

def _cm_eps(nc):
    """Clausius-Mossotti / Lorentz-Lorenz form, as a second independent nonlinear estimate of
    eps from nc alongside the plain Maxwell relation -- gives RidgeCV two distinct physical
    functional forms to blend instead of just a linear rescale of one."""
    n2 = np.clip(nc, 0, None) ** 2
    f = (n2 - 1.0) / (n2 + 2.0)
    return (1.0 + 2.0 * f) / np.clip(1.0 - f, 1e-6, None)

def _maxwell_nc(eps):
    return np.sqrt(np.clip(eps, 1e-6, None))

def _cm_nc(eps):
    f = (eps - 1.0) / (eps + 2.0)
    n2 = (1.0 + 2.0 * f) / np.clip(1.0 - f, 1e-6, None)
    return np.sqrt(np.clip(n2, 1e-6, None))

# Each recipe maps target -> (sibling targets, [list of derived-feature functions]). Every
# function's output becomes its own design-matrix column (evaluated only where every sibling
# is known); RidgeCV then learns how to weight/combine them, plus the raw sibling values
# themselves. eps/nc get two independent physical forms (Maxwell + Clausius-Mossotti) instead of
# a single identity, since on this data the plain Maxwell relation alone under-fit those targets.
PHYS_RECIPES = {
    "ei":  (["eea", "egc"], [lambda a, b: a + b]),          # Koopmans
    "eea": (["ei", "egc"],  [lambda a, b: a - b]),          # Koopmans
    "egc": (["ei", "eea"],  [lambda a, b: a - b]),          # Koopmans
    "egb": (["egc"],        [lambda a: a]),                 # bulk vs chain bandgap
    "eps": (["nc"],         [_maxwell_eps, _cm_eps]),       # Maxwell + Clausius-Mossotti
    "nc":  (["eps"],        [_maxwell_nc, _cm_nc]),
}
PHYS_MIN_COVERED = 25

_PLUT = {t: dict(zip(train.loc[train["target_type"] == t, "smiles_canon"],
                      train.loc[train["target_type"] == t, "target"]))
          for t in TARGETS}


def _phys_design(mols, sibs, fns):
    """Design matrix = [one column per derived-feature function in `fns`] + [raw sibling
    columns], evaluated on rows where every sibling is known."""
    S = np.full((len(mols), len(sibs)), np.nan, dtype=np.float64)
    for j, s in enumerate(sibs):
        lut = _PLUT.get(s, {})
        S[:, j] = [lut.get(m, np.nan) for m in mols]
    cov = np.isfinite(S).all(axis=1)
    n_derived = len(fns)
    Fd = np.full((len(mols), n_derived + len(sibs)), np.nan, dtype=np.float64)
    if cov.any():
        cols = [S[cov, j] for j in range(len(sibs))]
        for fi, fn in enumerate(fns):
            Fd[cov, fi] = fn(*cols)
        for j in range(len(sibs)):
            Fd[cov, n_derived + j] = cols[j]
    return Fd, cov


def train_physics_head(target):
    if target not in PHYS_RECIPES:
        return None
    sibs, fns = PHYS_RECIPES[target]
    if any(s not in TARGETS for s in sibs):
        return None
    mask = (train["target_type"].values == target)
    y = train.loc[mask, "target"].values.astype(np.float64)
    mols_tr = train.loc[mask, "smiles_canon"].values
    F_tr, cov_tr = _phys_design(mols_tr, sibs, fns)
    if cov_tr.sum() < PHYS_MIN_COVERED:
        print(f"[{target}] physics head: only {int(cov_tr.sum())} covered rows -> skipped")
        return None

    oof = np.full(len(y), np.nan)
    Fc, yc, gcv = F_tr[cov_tr], y[cov_tr], mols_tr[cov_tr]
    k = int(min(5, max(2, len(np.unique(gcv)) // 10)))
    for a, b in make_group_folds(gcv, k, seed=SEED):
        mdl = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0]).fit(Fc[a], yc[a])
        oof[np.where(cov_tr)[0][b]] = mdl.predict(Fc[b])
    r2_cov = r2_score(yc, oof[cov_tr])

    full = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0]).fit(Fc, yc)
    mols_te = test["smiles_canon"].values
    F_te, cov_te = _phys_design(mols_te, sibs, fns)
    tst = np.full(len(test), np.nan)
    if cov_te.any():
        tst[cov_te] = full.predict(F_te[cov_te])

    print(f"[{target}] physics head ({'+'.join(sibs)}): "
          f"train cov {int(cov_tr.sum())}/{len(y)} ({100 * cov_tr.mean():.1f}%), "
          f"test cov {int(cov_te.sum())}/{len(test)} ({100 * cov_te.mean():.1f}%), "
          f"OOF R2 on covered = {r2_cov:.4f}")
    return {"oof": oof, "cov_tr": cov_tr, "test": tst, "cov_te": cov_te, "r2_cov": r2_cov}


print("=" * 70); print("Calibrated physics heads (sibling delta-learning)"); print("=" * 70)
phys_res = {}
for t in TARGETS:
    r = train_physics_head(t)
    if r is not None: phys_res[t] = r


def apply_physics_gate(target, model_oof, model_test, y, valid_mask):
    """Blend the physics head into the GNN+CapsNet prediction, on covered rows only. alpha is
    chosen by group-fold CV over covered rows and includes 0, so a target with no covered gain
    keeps its GNN+CapsNet prediction unchanged. Returns (new_oof, new_test, alpha, gain)."""
    if target not in phys_res:
        return model_oof, model_test, 0.0, 0.0
    pr = phys_res[target]
    cov = pr["cov_tr"][valid_mask]
    if cov.sum() < PHYS_MIN_COVERED:
        return model_oof, model_test, 0.0, 0.0
    p_oof = pr["oof"][valid_mask]
    base_r2 = r2_score(y, model_oof)

    grid = np.linspace(0.0, 1.0, 21)
    mols = train.loc[(train["target_type"].values == target), "smiles_canon"].values[valid_mask]
    gcv = mols[cov]
    k = int(min(5, max(2, len(np.unique(gcv)) // 10)))
    cv_scores = np.zeros(len(grid))
    for a, b in make_group_folds(gcv, k, seed=SEED):
        ev = np.where(cov)[0][b]
        for gi, al in enumerate(grid):
            trial = model_oof.copy()
            trial[ev] = (1 - al) * model_oof[ev] + al * p_oof[ev]
            cv_scores[gi] += r2_score(y, trial)
    alpha = float(grid[int(np.argmax(cv_scores))])

    new_oof = model_oof.copy()
    new_oof[cov] = (1 - alpha) * model_oof[cov] + alpha * p_oof[cov]
    new_r2 = r2_score(y, new_oof)

    new_test = model_test.copy()
    ct = pr["cov_te"] & np.isfinite(pr["test"])
    if alpha > 0 and ct.any():
        new_test[ct] = (1 - alpha) * model_test[ct] + alpha * pr["test"][ct]
    gain = new_r2 - base_r2
    print(f"[{target}] physics gate: alpha={alpha:.2f}  OOF {base_r2:.4f} -> {new_r2:.4f} ({gain:+.4f})")
    return new_oof, new_test, alpha, gain


final = {}; oof_scores = {}
for t in TARGETS:
    if t not in final_res:
        # specialist training failed outright for this target (e.g. too few encodable
        # molecules or every fold fell below the minimum row count) -- fall back to the
        # training median rather than crashing the whole submission.
        print(f"[{t}] WARNING: no trained specialist -> falling back to training median")
        final[t] = np.full(len(test), train.loc[train["target_type"] == t, "target"].median())
        oof_scores[t] = float("nan")
        continue
    r = final_res[t]
    y_t, oof_t, test_t = r["y"], r["oof"], r["test"]
    valid = np.isfinite(oof_t)
    yv = y_t[valid]
    base_r2 = r2_score(yv, oof_t[valid])
    new_oof, new_test, alpha, gain = apply_physics_gate(t, oof_t[valid], test_t, yv, valid)
    final_r2 = r2_score(yv, new_oof)
    final[t] = new_test; oof_scores[t] = final_r2
    print(f"[{t}] GNN+CapsNet {base_r2:.4f} -> +physics gate {final_r2:.4f}\n")

mean_oof = float(np.nanmean(list(oof_scores.values())))
print("HONEST FINAL OOF (group-fold, single-model pipeline) per target:")
for t in TARGETS:
    val = f"{oof_scores[t]:.4f}" if np.isfinite(oof_scores[t]) else "N/A (median fallback)"
    print(f"   {t:>4}: {val}")
print(f"   MEAN R2 (competition metric, over targets with a trained model) = {mean_oof:.4f}")


## 9. Submission Generation

In [ ]:
pred = np.full(len(test), np.nan)
for t in TARGETS:
    m = (test["target_type"].values == t)
    pred[m] = final[t][m]

# safety net only: any non-finite prediction falls back to that target's training median
for t in TARGETS:
    m = (test["target_type"].values == t) & ~np.isfinite(pred)
    if m.any():
        pred[m] = train.loc[train["target_type"] == t, "target"].median()
        print(f"WARNING: filled {int(m.sum())} non-finite predictions for {t}")

sub = pd.DataFrame({"id": test["id"].values, "target": pred})
sub.to_csv("submission.csv", index=False)
print(f"\nSaved submission.csv {sub.shape}")
print(sub.head().to_string(index=False))
